In [21]:
import csv, json, os, urllib.parse, urllib.request

BASE = "https://www.consumerfinance.gov/data-research/consumer-complaints/search/api/v1/"
params = {
    "format": "json",
    "no_aggs": "true",
    "no_highlight": "true",
    "product": "Mortgage",
    "has_narrative": "true",
    "date_received_min": "2025-01-01",
    "date_received_max": "2026-03-01",
    "consumer_consent_provided": "Consent provided",
}
q = urllib.parse.urlencode(params)
with urllib.request.urlopen(f"{BASE}?{q}", timeout=300) as r:
    data = json.load(r)
if isinstance(data, dict) and data.get("error"):
    raise RuntimeError(data["error"])
rows = [h["_source"] for h in data]
if not rows:
    raise RuntimeError("No rows returned")

cols = ["complaint_id", "date_received", "issue", "sub_issue", "complaint_what_happened"]
rows_out = [{c: (r.get(c) or "") for c in cols} for r in rows]

os.makedirs("data", exist_ok=True)
out = "data/mortgage_2025_narrative_consent.csv"
with open(out, "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=cols)
    w.writeheader()
    w.writerows(rows_out)
len(rows_out), out

(15755, 'data/mortgage_2025_narrative_consent.csv')

In [22]:
import pandas as pd

mortgage_df = pd.read_csv("data/mortgage_2025_narrative_consent.csv")

In [23]:
mortgage_df.head()

,complaint_id,date_received,issue,sub_issue,complaint_what_happened
0,15844200,2025-09-10T12:00:00-05:00,Struggling to pay mortgage,Trying to communicate with the company to fix ...,"Dear CFPB, We are filing a formal complaint ag..."
1,16329523,2025-10-02T12:00:00-05:00,Trouble during payment process,Payment process,This company continues to keep reporting and m...
2,11466691,2025-01-10T12:00:00-05:00,Struggling to pay mortgage,Trying to communicate with the company to fix ...,I currently have an ongoing mortgage account w...
3,13427712,2025-05-09T12:00:00-05:00,Trouble during payment process,"Escrow, taxes, or insurance",I had XXXX mortgages with CMG financial. The f...
4,17365063,2025-11-19T12:00:00-05:00,Trouble during payment process,Payment process,I transferred {$300000.00} from my checking ac...


In [24]:
mortgage_df.count()

complaint_id               15755
date_received              15755
issue                      15755
sub_issue                  15744
complaint_what_happened    15755
dtype: int64